In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.0 MB/s eta 0:00:00


In [ ]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt
import optuna

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

dataset_path = '/content/drive/My Drive/dataset'
print(f"Listing contents of: {dataset_path}")



Listing contents of: /content/drive/My Drive/dataset


In [ ]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # this mean and std is chosen bcz this is of imagenet, as we do TL later, this will help
])

In [ ]:
dataset = datasets.ImageFolder(root = dataset_path, transform=image_transforms)
len(dataset)

2300

In [ ]:
num_classes = len(dataset.classes)
num_classes

6

In [ ]:
train_size = int(0.75*len(dataset))
val_size = len(dataset)-train_size

In [ ]:
from torch.utils.data import random_split

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

In [ ]:
#load pretrained Reset model
class CarClassifierResNet(nn.Module):
  def __init__(self, num_classes, dropout_rate= 0.5):
    super().__init__()
    self.model = models.resnet50(weights='DEFAULT')

    # Freeze all layers except final fc layer
    for param in self.model.parameters():
      param.requires_grad = False

    in_features = self.model.fc.in_features

    # Unfreeze Layer-4 and Replace final fc layer
    for param in self.model.layer4.parameters():
      param.requires_grad = True

    self.model.fc = nn.Sequential(
        nn.Dropout(dropout_rate),
        nn.Linear(in_features, num_classes)
    )
  def forward(self, x):
    x = self.model(x)
    return x




In [ ]:
# Define the objective function for Optuna
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)

    # Load the model
    model = CarClassifierResNet(num_classes=num_classes, dropout_rate=dropout_rate).to(device)

    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    # Training loop (using fewer epochs for faster hyperparameter tuning)
    epochs = 3
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total

        # Report intermediate result to Optuna
        trial.report(accuracy, epoch)

        # Handle pruning (if applicable)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    end = time.time()
    print(f"Execution time: {end - start} seconds")

    return accuracy

In [ ]:
# Create the study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2026-06-17 13:11:03,151] A new study created in memory with name: no-name-ee3118d2-3e12-43b7-a34b-8bf068be1c69


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 77.9MB/s]
[I 2026-06-17 14:02:09,023] Trial 0 finished with value: 76.34782608695652 and parameters: {'lr': 0.00017271107201738704, 'dropout_rate': 0.4358242531159978}. Best is trial 0 with value: 76.34782608695652.


Execution time: 3063.2092111110687 seconds


[I 2026-06-17 14:41:56,573] Trial 1 finished with value: 73.91304347826087 and parameters: {'lr': 6.385279205767341e-05, 'dropout_rate': 0.4321924818901708}. Best is trial 0 with value: 76.34782608695652.


...

Execution time: 2405.016463279724 seconds


[I 2026-06-17 16:02:03,695] Trial 3 finished with value: 77.56521739130434 and parameters: {'lr': 0.0032244110007983978, 'dropout_rate': 0.6303945447389249}. Best is trial 3 with value: 77.56521739130434.


Execution time: 2400.066709756851 seconds


[I 2026-06-17 16:42:25,113] Trial 4 finished with value: 72.52173913043478 and parameters: {'lr': 7.310225142614618e-05, 'dropout_rate': 0.2513641288305718}. Best is trial 3 with value: 77.56521739130434.


Execution time: 2420.7246301174164 seconds


In [ ]:
study.best_params